# <font color="#418FDE" size="6.5" uppercase>**Daten aufteilen**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Erstellen reproduzierbare Train-, Validierungs- und Testaufteilungen mit Indizes. 
- Berücksichtigen Klassen, Gruppen und Zeitordnung bei manuellen Splits. 
- Erkennen und dokumentieren typische Formen von Datenleckage. 


## **1. Split Grundlagen**

### **1.1. Warum Daten aufteilen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_01_01.jpg?v=1787635336" width="250">



>* Modelle sollen auf unbekannte Daten verallgemeinern
>* Train-, Validierungs- und Testdaten trennen Rollen

>* Training lernt, Validierung steuert Modellentscheidungen
>* Testdaten prüfen unabhängige Leistung auf neuen Fällen

>* Feste Splits machen Ergebnisse vergleichbar.
>* Indizes dokumentieren jede Datenzuordnung nachvollziehbar.



### **1.2. Indizes zufällig mischen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_01_02.jpg?v=1787635338" width="250">



>* Zufälliges Mischen verhindert verzerrte Datensplits
>* Gemischte Indizes erzeugen ähnlich verteilte Teilmengen

>* Indizes machen Splits nachvollziehbar und kontrollierbar
>* Zusammengehörige Daten bleiben korrekt getrennt

>* Feste Seeds machen Splits wiederholbar
>* Sonderfälle brauchen zusätzliche Splitregeln



In [ ]:
#@title Python-Code - Indizes zufällig mischen

# Wir mischen Zeilenindizes reproduzierbar für Datensplits.
# Der Seed macht die Zufallsreihenfolge wiederholbar.
# Die Ausgabe zeigt getrennte Indexmengen.

import numpy as np
import pandas as pd

# Ein kleiner Datensatz macht die Indizes gut sichtbar.
data = pd.DataFrame(
    {
        "student_id": range(1001, 1013),
        "score": [72, 85, 91, 64, 78, 88, 69, 95, 73, 81, 67, 90],
    }
)

# Diese Prüfung schützt vor unerwartet leeren Beispieldaten.
if len(data) != 12:
    raise ValueError("Der Beispieldatensatz sollte genau 12 Zeilen haben.")

# Der Generator erzeugt bei gleichem Seed dieselbe Mischung.
rng = np.random.default_rng(42)
shuffled_indices = rng.permutation(data.index.to_numpy())

# Die Schnittpunkte legen Train, Validierung und Test fest.
train_end = 7
validation_end = 9
train_indices = shuffled_indices[:train_end]

validation_indices = shuffled_indices[train_end:validation_end]
test_indices = shuffled_indices[validation_end:]

# Dieselbe Seed-Einstellung bestätigt die Reproduzierbarkeit.
check_rng = np.random.default_rng(42)
repeated_indices = check_rng.permutation(data.index.to_numpy())
is_reproducible = np.array_equal(shuffled_indices, repeated_indices)

# Die Daten bleiben unverändert, nur die Auswahlindizes wechseln.
train_students = data.loc[train_indices, "student_id"].to_list()
validation_students = data.loc[validation_indices, "student_id"].to_list()
test_students = data.loc[test_indices, "student_id"].to_list()

print("Originale Indizes: " + str(data.index.to_list()))
print("Gemischte Indizes: " + str(shuffled_indices.tolist()))
print("Train-Indizes: " + str(train_indices.tolist()))
print("Validierungs-Indizes: " + str(validation_indices.tolist()))
print("Test-Indizes: " + str(test_indices.tolist()))
print("Train-Studierende: " + str(train_students))
print("Validierungs-Studierende: " + str(validation_students))
print("Test-Studierende: " + str(test_students))
print("Reproduzierbar mit Seed 42: " + str(is_reproducible))



### **1.3. Splitgrößen prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_01_03.jpg?v=1787635339" width="250">



>* Tatsächliche Splitgrößen vor dem Training prüfen
>* Kleine Splits können Bewertungen verzerren

>* Jeder Split erfüllt eine eigene Aufgabe
>* Größen passend zur Datenvielfalt prüfen

>* Splitgrößen dokumentieren und Erwartungen prüfen
>* Fehler erkennen und Reproduzierbarkeit sichern



In [ ]:
#@title Python-Code - Splitgrößen prüfen

# Wir prüfen Splitgrößen nach einer reproduzierbaren Aufteilung.
# Indizes zeigen, welche Zeilen wohin gehören.
# Die Ausgabe bestätigt Anzahl, Summe und Überschneidungen.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Ein kleiner Datensatz macht Rundungseffekte gut sichtbar.
row_count = 37
indices = np.arange(row_count)
labels = np.array([0] * 25 + [1] * 12)

# Zuerst trennen wir den endgültigen Testbereich ab.
train_valid_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=labels
)

# Danach teilen wir den Rest in Training und Validierung.
train_idx, valid_idx = train_test_split(
    train_valid_idx, test_size=0.25, random_state=42, stratify=labels[train_valid_idx]
)

# Diese Tabelle dokumentiert die tatsächlichen Splitgrößen.
split_names = ["Training", "Validierung", "Test"]
split_indices = [train_idx, valid_idx, test_idx]
split_sizes = [len(part) for part in split_indices]

# Prozentwerte helfen beim Vergleich mit dem ursprünglichen Plan.
size_table = pd.DataFrame({"Split": split_names, "Zeilen": split_sizes})
size_table["Anteil"] = (size_table["Zeilen"] / row_count).round(3)

# Eine einfache Prüfung erkennt ausgelassene oder doppelte Indizes.
all_split_indices = np.concatenate(split_indices)
unique_count = len(np.unique(all_split_indices))
no_overlap = unique_count == row_count and len(all_split_indices) == row_count

print("Geplante Anteile: Training 60%, Validierung 20%, Test 20%")
print(size_table.to_string(index=False))
print(f"Gesamtzahl geprüft: {sum(split_sizes)} von {row_count} Zeilen")
print(f"Keine Überschneidungen oder Lücken: {no_overlap}")



## **2. Spezielle Splits**

### **2.1. Klassen ausgewogen aufteilen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_02_01.jpg?v=1787635341" width="250">



>* Klassenanteile in allen Splits ähnlich halten
>* Seltene Klassen besonders sorgfältig berücksichtigen

>* Stratifizierung erhält Klassenanteile in allen Splits
>* Seltene Klassen werden zuverlässiger bewertet

>* Kleine Klassen transparent dokumentieren
>* Gruppen, Zeitordnung und Leckage mitbedenken



In [ ]:
#@title Python-Code - Klassen ausgewogen aufteilen

# Wir vergleichen zufällige und stratifizierte Klassensplits.
# Die Klassenanteile sollen in Teilmengen ähnlich bleiben.
# Die Grafik zeigt stabilere Anteile durch Stratifikation.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Ein kleiner Datensatz mit unausgewogenen Klassen entsteht deterministisch.
labels = np.array(["gesund"] * 80 + ["krank"] * 20)
indices = np.arange(labels.size)

# Diese Prüfung macht die Annahme des Beispiels sichtbar.
if labels.size != indices.size:
    raise ValueError("Labels und Indizes müssen gleich lang sein.")

# Zuerst teilen wir ohne Beachtung der Klassen auf.
random_train, random_temp, y_random_train, y_random_temp = train_test_split(
    indices, labels, test_size=0.4, random_state=42
)

# Danach teilen wir den Rest erneut zufällig auf.
random_valid, random_test, y_random_valid, y_random_test = train_test_split(
    random_temp, y_random_temp, test_size=0.5, random_state=42
)

# Nun wiederholen wir beide Schritte mit Stratifikation.
strat_train, strat_temp, y_strat_train, y_strat_temp = train_test_split(
    indices, labels, test_size=0.4, stratify=labels, random_state=42
)

# Auch Validierung und Test behalten die Klassenanteile bei.
strat_valid, strat_test, y_strat_valid, y_strat_test = train_test_split(
    strat_temp, y_strat_temp, test_size=0.5, stratify=y_strat_temp, random_state=42
)

# Diese Hilfsfunktion berechnet den Anteil der seltenen Klasse.
def sick_share(values):
    return round(100 * np.mean(values == "krank"), 1)

# Die Ergebnisse werden kompakt für Tabelle und Grafik gesammelt.
summary = pd.DataFrame(
    {
        "Split": ["Training", "Validierung", "Test"],
        "Zufällig": [sick_share(y_random_train), sick_share(y_random_valid), sick_share(y_random_test)],
        "Stratifiziert": [sick_share(y_strat_train), sick_share(y_strat_valid), sick_share(y_strat_test)],
    }
)

# Die Tabelle zeigt die wichtigsten Zahlen ohne den ganzen Datensatz.
print("Anteil der Klasse 'krank' in Prozent:")
print(summary.to_string(index=False))
print("Stratifizierung hält die Klassenanteile näher am Gesamtwert von 20%.")

# Eine einzelne Grafik macht den Unterschied schnell sichtbar.
ax = summary.plot(
    x="Split", y=["Zufällig", "Stratifiziert"], kind="bar", figsize=(7, 4)
)

# Beschriftungen erklären, was die Balkenhöhen bedeuten.
ax.set_title("Klassenanteile nach Split-Methode")
ax.set_xlabel("Teilmenge")
ax.set_ylabel("Anteil 'krank' in Prozent")
ax.axhline(20, color="gray", linestyle="--", linewidth=1, label="Gesamtanteil")

# Die Legende hilft beim Vergleich der Methoden.
ax.legend(title="Methode")
plt.tight_layout()
plt.show()



### **2.2. Gruppen sauber trennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_02_02.jpg?v=1787635343" width="250">



>* Gleiche Gruppen nie über Splits verteilen
>* Sonst wirkt die Testleistung zu optimistisch

>* Passende Gruppeneinheit fachlich bestimmen
>* Datenleckage und unausgewogene Splits vermeiden

>* Gruppensplits mit Klassenverteilung gemeinsam prüfen
>* Abweichungen dokumentieren für glaubwürdige Evaluation



In [ ]:
#@title Python-Code - Gruppen sauber trennen

# Dieses Beispiel zeigt saubere Gruppentrennung.
# Gruppen dürfen keine Splitgrenzen überschreiten.
# Die Ausgabe vergleicht falsche und richtige Splits.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit

# Wir erzeugen kleine Messdaten mit wiederholten Gruppen.
rng = np.random.default_rng(42)
groups = np.repeat(np.arange(1, 13), 4)

# Jede Gruppe erhält eine feste Klasse.
group_labels = np.array([0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0])
labels = np.repeat(group_labels, 4)

# Eine Messung ähnelt anderen Messungen derselben Gruppe.
measurement = groups * 10 + rng.normal(0, 1, size=len(groups))
data = pd.DataFrame({"group": groups, "label": labels, "measurement": measurement})

# Ein zufälliger Zeilensplit trennt Gruppen oft nicht sauber.
row_order = rng.permutation(len(data))
row_test_index = row_order[:12]
row_train_index = row_order[12:]

# Ein Gruppensplit wählt ganze Gruppen für den Testbereich.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
group_train_index, group_test_index = next(splitter.split(data, labels, groups))

# Diese Funktion zählt Gruppen, die in beiden Bereichen vorkommen.
def count_overlapping_groups(train_index, test_index):
    train_groups = set(data.loc[train_index, "group"])
    test_groups = set(data.loc[test_index, "group"])
    return len(train_groups.intersection(test_groups))

# Diese Funktion fasst Größe und Klassenanteil kurz zusammen.
def summarize_split(name, train_index, test_index):
    overlap = count_overlapping_groups(train_index, test_index)
    test_positive_rate = data.loc[test_index, "label"].mean()
    print(f"{name}: Testzeilen={len(test_index)}, überlappende Gruppen={overlap}")
    print(f"{name}: positiver Klassenanteil im Test={test_positive_rate:.2f}")

print(f"Datensatz: {len(data)} Zeilen aus {data['group'].nunique()} Gruppen.")
summarize_split("Zeilensplit", row_train_index, row_test_index)
summarize_split("Gruppensplit", group_train_index, group_test_index)

# Für die Grafik markieren wir die Testzeilen beider Strategien.
data["row_split"] = "Training"
data.loc[row_test_index, "row_split"] = "Test"

# Beim Gruppensplit liegen ganze Gruppen im Testbereich.
data["group_split"] = "Training"
data.loc[group_test_index, "group_split"] = "Test"

# Die x-Achse zeigt Gruppen, die y-Achse die Splitstrategie.
fig, ax = plt.subplots(figsize=(8, 3))
colors = {"Training": "tab:blue", "Test": "tab:orange"}

for y_value, column in enumerate(["row_split", "group_split"]):
    for split_name, color in colors.items():
        subset = data[data[column] == split_name]
        ax.scatter(subset["group"], np.full(len(subset), y_value), c=color, label=split_name)

ax.set_yticks([0, 1])
ax.set_yticklabels(["Zeilensplit", "Gruppensplit"])
ax.set_xlabel("Gruppen-ID")
ax.set_ylabel("Splitstrategie")

ax.set_title("Testpunkte: zufällig nach Zeilen oder sauber nach Gruppen")
handles, labels_for_legend = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels_for_legend[:2], loc="upper right")
plt.show()



### **2.3. Zeitordnung beachten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_02_03.jpg?v=1787635345" width="250">



>* Zeitliche Reihenfolge beim Splitten bewahren
>* Vergangenheit trainiert, Zukunft validiert und testet

>* Zeitdaten nicht zufällig mischen
>* Chronologische Splits verhindern Datenleckage

>* Mehrere Validierungszeiträume prüfen stabile Modellleistung
>* Zeitgrenzen und besondere Ereignisse dokumentieren



In [ ]:
#@title Python-Code - Zeitordnung beachten

# Dieses Beispiel zeigt chronologische Datenaufteilungen.
# Zeitordnung verhindert unrealistische Zukunftsinformationen.
# Die Grafik markiert Training, Validierung und Test.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wir erzeugen kleine, zeitlich geordnete Beispieldaten.
months = pd.date_range("2023-01-01", periods=24, freq="MS")
trend = np.linspace(100, 150, len(months))
season = np.array([0, 4, 8, 6, 2, -2, -5, -3, 1, 5, 7, 3] * 2)
sales = trend + season

# Die Indizes bleiben in ihrer natürlichen Reihenfolge.
data = pd.DataFrame({"month": months, "sales": sales})
train_end = 14
valid_end = 19

# Jeder Zeitraum bekommt genau eine Rolle.
data["split"] = "Test"
data.loc[:train_end - 1, "split"] = "Training"
data.loc[train_end:valid_end - 1, "split"] = "Validierung"

# Diese Prüfung schützt vor versehentlich vertauschten Zeiträumen.
if not data["month"].is_monotonic_increasing:
    raise ValueError("Die Monate müssen chronologisch sortiert sein.")

# Wir fassen die gewählten Zeiträume kurz zusammen.
summary = data.groupby("split", sort=False)["month"].agg(["min", "max", "count"])
summary["min"] = summary["min"].dt.strftime("%Y-%m")
summary["max"] = summary["max"].dt.strftime("%Y-%m")

print("Chronologische Aufteilung mit Indizes:")
for split_name, row in summary.iterrows():
    print(f"{split_name}: {row['min']} bis {row['max']} ({row['count']} Monate)")
print("Wichtig: Kein späterer Monat liegt im Training.")

# Die Farben machen die zeitliche Trennung sichtbar.
colors = {"Training": "tab:blue", "Validierung": "tab:orange", "Test": "tab:green"}
fig, ax = plt.subplots(figsize=(9, 4))

for split_name in ["Training", "Validierung", "Test"]:
    part = data[data["split"] == split_name]
    ax.scatter(part["month"], part["sales"], label=split_name, color=colors[split_name])

ax.plot(data["month"], data["sales"], color="lightgray", zorder=0)
ax.set_title("Zeitordnung beachten: erst Vergangenheit, dann Zukunft")
ax.set_xlabel("Monat")
ax.set_ylabel("Absatz in Stück")
ax.legend()
plt.show()



## **3. Datenleckage erkennen**

### **3.1. Zielwertleckage erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_03_01.jpg?v=1787635331" width="250">



>* Zielwert steckt unzulässig in Modellmerkmalen
>* Leckage versteckt sich oft in späteren Daten

>* Prüfe Verfügbarkeit zum Vorhersagezeitpunkt
>* Achte auf zukünftige und aggregierte Informationen

>* Verdächtige Variablen und Entscheidungen dokumentieren
>* Einsatzkontext klären, Leckagen sorgfältig prüfen



In [ ]:
#@title Python-Code - Zielwertleckage erkennen

# Dieses Beispiel zeigt Zielwertleckage in Merkmalen.
# Ein verdächtiges Merkmal verrät den Zielwert.
# Die Testgenauigkeit wirkt dadurch unrealistisch hoch.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Wir erzeugen kleine Kundendaten mit bekanntem Zielwert.
rng = np.random.default_rng(42)
n_samples = 300
usage_drop = rng.normal(0.0, 1.0, n_samples)
complaints = rng.poisson(1.2, n_samples)

# Der Zielwert hängt nur teilweise von frühen Signalen ab.
risk_score = usage_drop + 0.7 * complaints + rng.normal(0.0, 1.2, n_samples)
churn = (risk_score > 1.8).astype(int)

# Diese Spalte entsteht erst nach der Kündigung.
retention_status = np.where(churn == 1, "gekündigt", "aktiv")
leaky_flag = (retention_status == "gekündigt").astype(int)

# Wir bauen zwei Datensätze: sauber und undicht.
clean_features = pd.DataFrame(
    {"usage_drop": usage_drop, "complaints": complaints}
)
leaky_features = clean_features.copy()
leaky_features["retention_status_flag"] = leaky_flag

# Die gleiche Aufteilung macht den Vergleich fair.
clean_train, clean_test, y_train, y_test = train_test_split(
    clean_features, churn, test_size=0.3, stratify=churn, random_state=42
)
leaky_train = leaky_features.loc[clean_train.index]
leaky_test = leaky_features.loc[clean_test.index]

# Eine einfache Prüfung schützt vor stillen Datenfehlern.
if len(clean_train) != len(leaky_train):
    raise ValueError("Die Trainingsdaten passen nicht zusammen.")

# Wir trainieren dasselbe Modell einmal sauber und einmal undicht.
clean_model = LogisticRegression(random_state=42, max_iter=200)
clean_model.fit(clean_train, y_train)
leaky_model = LogisticRegression(random_state=42, max_iter=200)
leaky_model.fit(leaky_train, y_train)

# Die undichte Spalte macht die Auswertung künstlich perfekt.
clean_accuracy = accuracy_score(y_test, clean_model.predict(clean_test))
leaky_accuracy = accuracy_score(y_test, leaky_model.predict(leaky_test))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Saubere Testgenauigkeit: {clean_accuracy:.3f}")
print(f"Mit Zielwertleckage: {leaky_accuracy:.3f}")
print("Warnsignal: Ein nachträglicher Status erklärt den Zielwert direkt.")

# Das Balkendiagramm macht den Unterschied sichtbar.
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["saubere Merkmale", "mit Leckage"], [clean_accuracy, leaky_accuracy])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Testgenauigkeit")
ax.set_title("Zielwertleckage erzeugt überoptimistische Metriken")
plt.show()



### **3.2. Skalierung ohne Leckage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_03_02.jpg?v=1787635332" width="250">



>* Skalierung nur mit Trainingsdaten anpassen
>* Testdaten können Ergebnisse sonst verfälschen

>* Skalierung nur mit Trainingsdaten anpassen
>* Validierungs- und Testdaten unverändert transformieren

>* Transformationen nur innerhalb des Trainingssplits lernen
>* Skalierungsdaten und Ausschlüsse sauber dokumentieren



In [ ]:
#@title Python-Code - Skalierung ohne Leckage

# Dieses Beispiel zeigt Skalierung ohne Datenleckage.
# Skalierungswerte dürfen nur aus Trainingsdaten stammen.
# Die Ausgabe vergleicht falsche und korrekte Skalierung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine Messwerte mit einem auffälligen Testbereich.
rng = np.random.default_rng(42)
normal_values = rng.normal(loc=50.0, scale=8.0, size=36)
outlier_values = np.array([95.0, 100.0, 105.0, 110.0])

# Ein Merkmal reicht, um den Leckageeffekt sichtbar zu machen.
feature_values = np.concatenate([normal_values, outlier_values])
X = feature_values.reshape(-1, 1)
y = (feature_values > 60.0).astype(int)

# Die Aufteilung ist reproduzierbar und nutzt feste Indizes.
indices = np.arange(len(X))
train_idx, test_idx = train_test_split(
    indices, test_size=0.25, random_state=42, stratify=y
)

# Diese Prüfung macht die erwartete Datenform explizit.
if X.shape != (40, 1):
    raise ValueError("Erwartet werden 40 Zeilen und ein Merkmal.")

# Falsch: Der Skalierer sieht vorab alle Daten.
leaky_scaler = StandardScaler()
X_all_scaled = leaky_scaler.fit_transform(X)
leaky_test_mean = X_all_scaled[test_idx].mean()

# Richtig: Der Skalierer lernt nur aus Trainingsdaten.
clean_scaler = StandardScaler()
X_train_scaled = clean_scaler.fit_transform(X[train_idx])
X_test_scaled = clean_scaler.transform(X[test_idx])
clean_test_mean = X_test_scaled.mean()

# Wir vergleichen die gelernten Mittelwerte der Skalierer.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsmittelwert korrekt gelernt: {clean_scaler.mean_[0]:.2f}")
print(f"Mittelwert bei Leckage aus allen Daten: {leaky_scaler.mean_[0]:.2f}")
print(f"Testmittel nach falscher Skalierung: {leaky_test_mean:.2f}")
print(f"Testmittel nach korrekter Skalierung: {clean_test_mean:.2f}")

# Die Grafik zeigt, wie Testwerte unterschiedlich verschoben werden.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X[test_idx, 0], X_all_scaled[test_idx, 0], label="mit Leckage")
ax.scatter(X[test_idx, 0], X_test_scaled[:, 0], label="ohne Leckage")
ax.set_title("Testdaten: Skalierung mit und ohne Leckage")
ax.set_xlabel("Ursprünglicher Messwert")
ax.set_ylabel("Skalierter Wert")
ax.legend()
plt.show()



### **3.3. Split Protokoll**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_03_03.jpg?v=1787635334" width="250">



>* Splitentscheidungen nachvollziehbar dokumentieren
>* Datenleckage durch Gruppenüberschneidungen vermeiden

>* Vorverarbeitung nur auf Trainingsdaten lernen
>* Gelernte Transformationen im Protokoll dokumentieren

>* Zeit, Gruppen und Zieldefinition sauber dokumentieren
>* Leckage vermeiden, Bewertung glaubwürdig machen



In [ ]:
#@title Python-Code - Split Protokoll

# Dieses Beispiel erstellt ein kurzes Split Protokoll.
# Es zeigt typische Leckagefragen vor dem Modelltraining.
# Am Ende steht eine prüfbare Dokumentation.

import numpy as np
import pandas as pd

# Wir bauen kleine Beispieldaten mit Gruppen und Zeit.
records = pd.DataFrame(
    {
        "row_id": np.arange(12),
        "patient_id": [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6],
        "month": [1, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 6],
        "risk_score": [0.2, 0.3, 0.8, 0.7, 0.4, 0.5, 0.9, 0.8, 0.1, 0.2, 0.6, 0.7],
        "target": [0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1],
    }
)

# Diese Aufteilung ist absichtlich schlecht dokumentierbar.
train_bad = records.iloc[:8].copy()
test_bad = records.iloc[6:].copy()

# Diese Aufteilung hält Gruppen vollständig getrennt.
train_groups = [1, 2, 3, 4]
test_groups = [5, 6]

# Wir erzeugen saubere Trainings- und Testdaten.
train_good = records[records["patient_id"].isin(train_groups)].copy()
test_good = records[records["patient_id"].isin(test_groups)].copy()

# Eine einfache Prüfung findet Gruppenleckage.
bad_overlap = set(train_bad["patient_id"]) & set(test_bad["patient_id"])
good_overlap = set(train_good["patient_id"]) & set(test_good["patient_id"])

# Das Protokoll fasst die wichtigsten Splitentscheidungen zusammen.
protocol = pd.DataFrame(
    {
        "Prüfpunkt": ["Gruppen getrennt", "Zeitordnung notiert", "Skalierung gelernt"],
        "Schlechter Split": [len(bad_overlap) == 0, "nein", "vor Split"],
        "Sauberer Split": [len(good_overlap) == 0, "ja", "nur Training"],
    }
)

# Wir geben nur kurze, relevante Ergebnisse aus.
print("Split Protokoll: Leckageprüfung")
print("Schlechter Split, gemeinsame Patientinnen:", sorted(bad_overlap))
print("Sauberer Split, gemeinsame Patientinnen:", sorted(good_overlap))
print(protocol.to_string(index=False))
print("Merksatz: Jede Splitentscheidung muss vor der Bewertung nachvollziehbar sein.")



# <font color="#418FDE" size="6.5" uppercase>**Daten aufteilen**</font>


In this lecture, you learned to:
- Erstellen reproduzierbare Train-, Validierungs- und Testaufteilungen mit Indizes. 
- Berücksichtigen Klassen, Gruppen und Zeitordnung bei manuellen Splits. 
- Erkennen und dokumentieren typische Formen von Datenleckage. 

In the next Lecture (Lecture B), we will go over 'Verluste und Metriken'